# 1. Fields and domains

The data model of the prototype: dimensions, domains and fields, as defined in `concepts.py` and implemented by `NumPyArrayField` in `implementation.py`.

A **field** maps every point of its **domain** to a **value** of its **codomain**, e.g. `float`. The pairs `(point, value)` form its *graph* (see the "Semantics of domains, values and fields" notebook). The values never move: operations on fields compute a new domain and new values point by point.

Contents: dimensions, kinds and domains · fields and arithmetic · restriction and writing · premap by offsets · index sets and broadcasting · values.

## Dimensions, kinds and domains

| Concept | Kind | Meaning |
|---|---|---|
| `DimensionIndex` | class | An integer position along a dimension. A dimension is declared by subclassing a kind: `class I(Axis): ...` |
| `Axis` | kind | A Cartesian dimension, e.g. `I` or the vertical `K`: ranges only; offsets. An `Axis` has a staggered partner, at its half-integer positions, e.g. the half levels `Staggered[K]` (see "11. Staggered axes") |
| `Location` | kind | An unstructured dimension, the locations of a mesh of one kind, e.g. `Cell`: ranges or index sets; no offsets; the source or target of relations (see "2. Connectivities") |
| `Label` | kind | A finite dimension without topology, e.g. `Tracer`: ranges or index sets; no offsets; not in relations |
| `Neighbor` | kind | The fourth kind, built from relations: the neighbors of each source, a position in a row of the table (see "2. Connectivities") |
| `Dimension` | alias | `type[DimensionIndex]`: a dimension itself, e.g. `I` |
| `IndexSet` | alias | `tuple[int, ...]`, sorted and without duplicates: indices that need not be consecutive |
| `Domain` | alias | `dict[Dimension, range \| IndexSet]`: the points along each dimension; the domain is all their combinations |
| `check_domain` | function | Raises a `ValueError` if a domain breaks the rules of the kinds of its dimensions |
| `Offset` | alias | `tuple[Dimension, int]`: a shift along an `Axis`, written `I + 1` or `I - 1`; half offsets to a staggered partner, `K + 0.5`, are in "11. Staggered axes" |
| `Dims[*Ds]` | alias | The dimensions of a field, in order, for annotations: `Dims[I, J]` |
| `Key` | alias | A field subscript: dimension slices like `I.at(1):I.at(3)`, a `Domain`, or `...` for everything |

`I.at(3)` is the position 3 along `I`, and `I.at(0, 4)` is `range(0, 4)`. Index sets are meant for *unstructured* dimensions like `Cell`, whose points are only related through neighbor tables (see "2. Connectivities"), and for labels. Index sets on an `Axis` or a `Neighbor`, and unsorted ones, raise a `ValueError`. `Cell + 1` raises a `TypeError`, at runtime only: all the kinds share a metaclass, so type checkers accept it. A plain `DimensionIndex` subclass is legacy: allowed, and unchecked.

In [1]:
import numpy as np
from concepts import Axis, DimensionIndex, Label, Location, check_domain


class I(Axis): ...


class J(Axis): ...


class Cell(Location): ...


class Tracer(Label): ...  # e.g. the tracers of a model


class Old(DimensionIndex): ...  # legacy: no kind, unchecked


print("I.at(3) =", I.at(3), "| an int:", I.at(3) + 1 == 4, "| along I:", type(I.at(3)).__name__)
print("I.at(0, 4) =", I.at(0, 4))
dim, offset = J - 2  # an `Offset`
print("I + 1 == (I, 1):", I + 1 == (I, 1), "| J - 2:", dim.__name__, offset)

grid = {I: I.at(0, 4), J: J.at(0, 3)}  # 4 x 3 points
some_cells = {Cell: (0, 2, 5)}  # an index set: cells 0, 2 and 5
print("grid:", {d.__name__: r for d, r in grid.items()}, "| some cells:", some_cells[Cell])

try:
    Cell + 1  # a `Location` has no offsets
except TypeError as error:
    print("TypeError:", error)
try:
    check_domain({I: (0, 2)})  # an `Axis` has ranges only
except ValueError as error:
    print("ValueError:", error)
print("legacy: Old + 1 == (Old, 1):", Old + 1 == (Old, 1))

I.at(3) = I(3) | an int: True | along I: I
I.at(0, 4) = range(0, 4)
I + 1 == (I, 1): True | J - 2: J -2
grid: {'I': range(0, 4), 'J': range(0, 3)} | some cells: (0, 2, 5)
TypeError: 'Cell' has no offsets: only `Axis` dimensions have them.
ValueError: Index sets are not allowed along 'I'.
legacy: Old + 1 == (Old, 1): True


## Fields and arithmetic

`Field[D, T]` is a pure interface, without storage:

- `domain`: the `Domain` of the field. `D` (`Dims[...]`) and the codomain `T` are only used in annotations: `Field[Dims[I, J], float]`.
- `+`, `-`, `*` with fields or scalars: point by point. The domain of the result is computed *forward*, from the domains of the operands: their common dimensions are **intersected**, and the result has the dimensions of the left operand, then the missing ones of the right operand (**broadcasting**, see "Index sets and broadcasting").

`NDArrayField[D, T]` adds `data`, an `NDArray` with one axis per dimension, in the order of the domain. `NumPyArrayField(domain, data)` implements it with a numpy array. `Field` and `NDArrayField` are `runtime_checkable`: `isinstance` checks the member names.

In [2]:
from concepts import Field, NDArrayField
from implementation import NumPyArrayField

temperature = NumPyArrayField(grid, 280.0 + np.arange(12.0).reshape(4, 3))
anomaly = NumPyArrayField({I: I.at(1, 4), J: J.at(0, 3)}, np.ones((3, 3)))  # no row I(0)

print("temperature =", temperature)
print(
    "Field:",
    isinstance(temperature, Field),
    "| NDArrayField:",
    isinstance(temperature, NDArrayField),
)
print("data:", type(temperature.data).__name__, temperature.data.shape)
print("anomaly     =", anomaly)
print("minus 280   =", temperature - 280.0)
print("+ anomaly   =", temperature + anomaly)  # on the intersection: I[1:4]
print("2 * anomaly =", 2 * anomaly)

temperature = Field(I[0:4], J[0:3]: [[280.0, 281.0, 282.0], [283.0, 284.0, 285.0], [286.0, 287.0, 288.0], [289.0, 290.0, 291.0]])
Field: True | NDArrayField: True
data: ndarray (4, 3)
anomaly     = Field(I[1:4], J[0:3]: [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0], [1.0, 1.0, 1.0]])
minus 280   = Field(I[0:4], J[0:3]: [[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0], [9.0, 10.0, 11.0]])
+ anomaly   = Field(I[1:4], J[0:3]: [[284.0, 285.0, 286.0], [287.0, 288.0, 289.0], [290.0, 291.0, 292.0]])
2 * anomaly = Field(I[1:4], J[0:3]: [[2.0, 2.0, 2.0], [2.0, 2.0, 2.0], [2.0, 2.0, 2.0]])


## Restriction and writing

- `f[key]` restricts `f` to a sub-domain, with any `Key`: slices of dimension positions, a `Domain`, or `...`. Dimensions not in the key keep their whole domain.
- `f[key] = value` writes into a sub-domain of an `NDArrayField`: a scalar, or the values of a field on that sub-domain (the field may be larger).

A key selecting points outside the domain raises a `ValueError`, for reading and writing; keys on other dimensions are ignored.

In [3]:
print("rows 1-2      =", temperature[I.at(1) : I.at(3)])
print("one column    =", temperature[J.at(2) : J.at(3)])
print("a Domain      =", temperature[{I: I.at(2, 4), J: J.at(0, 2)}])
print("everything    =", temperature[...].domain == temperature.domain)
try:
    temperature[I.at(3) : I.at(5)]  # I(4) is outside the domain
except ValueError as error:
    print("ValueError:", error)
print("other dims    =", temperature[{Cell: (0,)}].domain == temperature.domain)  # ignored

out = NumPyArrayField(grid, np.zeros((4, 3)))
out[I.at(0) : I.at(1)] = -1.0  # a scalar
out[{I: I.at(1, 4), J: J.at(1, 3)}] = temperature  # only these values of `temperature` are read
print("out           =", out)

rows 1-2      = Field(I[1:3], J[0:3]: [[283.0, 284.0, 285.0], [286.0, 287.0, 288.0]])
one column    = Field(I[0:4], J[2:3]: [[282.0], [285.0], [288.0], [291.0]])
a Domain      = Field(I[2:4], J[0:2]: [[286.0, 287.0], [289.0, 290.0]])
everything    = True
ValueError: I[3:5] is not within the domain I[0:4], J[0:3].
other dims    = True
out           = Field(I[0:4], J[0:3]: [[-1.0, -1.0, -1.0], [0.0, 284.0, 285.0], [0.0, 287.0, 288.0], [0.0, 290.0, 291.0]])


## Premap by offsets

`f(I + 1)` is the field whose value at `I(i)` is the value of `f` at `I(i + 1)`: a **premap**, which only changes the domain (shifted by `-1` along `I`); the values stay the same. Several offsets apply in order: `f(I + 1, J - 1)`. Combined with arithmetic, premaps give stencils, whose domain is computed forward: a centered difference along `I` is defined where both neighbors are.

Fields can also be premapped by relations, on unstructured meshes: see "2. Connectivities".

In [4]:
from implementation import format_domain

east = temperature(I + 1)
print("temperature(I + 1)  =", east)  # same values, domain I[-1:3]
print("(I + 1, J - 1)      =", format_domain(temperature(I + 1, J - 1).domain))
gradient = 0.5 * (temperature(I + 1) - temperature(I - 1))
print("centered difference =", gradient)  # only I[1:3]: both neighbors exist

temperature(I + 1)  = Field(I[-1:3], J[0:3]: [[280.0, 281.0, 282.0], [283.0, 284.0, 285.0], [286.0, 287.0, 288.0], [289.0, 290.0, 291.0]])
(I + 1, J - 1)      = I[-1:3], J[1:4]
centered difference = Field(I[1:3], J[0:3]: [[3.0, 3.0, 3.0], [3.0, 3.0, 3.0]])


## Index sets and broadcasting

On an unstructured dimension, the points of interest are often not consecutive, e.g. the cells around a few edges. A domain can then be an `IndexSet` along that dimension: the values are stored in the order of the indices, which must be sorted, without duplicates (else a `ValueError`). Restriction, writing and arithmetic work the same: the intersection of a range and an index set is an index set (a `range` if its indices are consecutive). `index_set(...)` from `implementation.py` sorts indices, removes duplicates, and returns a `range` when they are consecutive; computed domains use it.

A `Label` dimension, e.g. `Tracer`, takes index sets too: below, two of the tracers of each cell. Fields with different dimensions are **broadcast**: the common dimensions are intersected, and the result has the dimensions of the left operand, then the missing ones of the right operand.

In [5]:
from implementation import index_set

pressure = NumPyArrayField({Cell: range(0, 6)}, [1000.0, 990.0, 985.0, 1010.0, 1005.0, 995.0])
selected = pressure[some_cells]
print("pressure       =", pressure)
print("selected cells =", selected, "| data", selected.data.tolist())

correction = NumPyArrayField({Cell: (2, 3, 5)}, [1.0, 2.0, 3.0])
print("+ correction   =", pressure + correction)  # on the intersection: cells 2, 3, 5
print("intersection   =", (selected + correction).domain[Cell])  # (0, 2, 5) & (2, 3, 5)

pressure[{Cell: (1, 4)}] = 0.0
print("written        =", pressure)
print("index_set:", index_set([5, 2, 2, 0]), index_set([4, 1, 3, 2]))
try:
    NumPyArrayField({Cell: (2, 0)}, [1.0, 2.0])  # not sorted
except ValueError as error:
    print("ValueError:", error)

mix = NumPyArrayField({Cell: range(2), Tracer: (0, 2)}, [[1.0, 2.0], [3.0, 4.0]])  # tracers 0, 2
per_cell = NumPyArrayField({Cell: range(3)}, [10.0, 20.0, 30.0])
per_tracer = NumPyArrayField({Tracer: (0, 2)}, [1.0, 2.0])
# (Cell, Tracer) on Cell[0:2]: [[1 * 10, 2 * 10], [3 * 20, 4 * 20]]
print("mix * per_cell        =", mix * per_cell)
# (Tracer, Cell): the dimensions of the left operand first, [[10, 20, 30], [20, 40, 60]]
print("per_tracer * per_cell =", per_tracer * per_cell)

pressure       = Field(Cell[0:6]: [1000.0, 990.0, 985.0, 1010.0, 1005.0, 995.0])
selected cells = Field(Cell{0, 2, 5}: [1000.0, 985.0, 995.0]) | data [1000.0, 985.0, 995.0]
+ correction   = Field(Cell{2, 3, 5}: [986.0, 1012.0, 998.0])
intersection   = (2, 5)
written        = Field(Cell[0:6]: [1000.0, 0.0, 985.0, 1010.0, 0.0, 995.0])
index_set: (0, 2, 5) range(1, 5)
ValueError: Index sets must be sorted, without duplicates, along 'Cell'.
mix * per_cell        = Field(Cell[0:2], Tracer{0, 2}: [[10.0, 20.0], [60.0, 80.0]])
per_tracer * per_cell = Field(Tracer{0, 2}, Cell[0:3]: [[10.0, 20.0, 30.0], [20.0, 40.0, 60.0]])


## Values

The values DSL operators take and return (simplified; only used by type checkers):

| Concept | Kind | Meaning |
|---|---|---|
| `Scalar` | alias | `float \| int` |
| `NumericValue` | alias | `Scalar \| Field` |
| `FunctionValue` | alias | Any callable, e.g. another DSL operator |
| `PrimitiveValue` | alias | `NumericValue \| FunctionValue` |
| `NestedTuple[T]` | alias | A tuple of `T`s and of nested tuples of them |
| `NamedCollection` | protocol | Any dataclass instance (its field values should be `AnyValue`s; not checked) |
| `CompositeValue` | alias | `NestedTuple[PrimitiveValue] \| NamedCollection` |
| `AnyValue` | alias | `PrimitiveValue \| CompositeValue` |

`dims_of` returns the dimensions of a field annotation; the DSL operators use it to read the annotations of their definitions. Composite values used by DSL operators: see "3. DSL definitions and operators".

In [6]:
from typing import get_type_hints

from concepts import Dims, dims_of


def heating(t: Field[Dims[I, J], float], rate: float) -> Field[Dims[I, J], float]: ...


hints = get_type_hints(heating)
print("dims of t:", [d.__name__ for d in dims_of(hints["t"])], "| rate:", hints["rate"].__name__)

dims of t: ['I', 'J'] | rate: float
